In [1]:
behavioral_features = [
    "log_TransactionAmt",
    "TransactionHour",
    "TransactionDay",
    "has_identity",
    "time_since_previous",
    "amount_change",
    "abs_amount_change",
    "card1_frequency",
    "card1_avg_previous_amount",
    "amount_vs_card_avg",
    "recent_card_transactions"
]

In [8]:
import gc
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Load transaction data
# ---------------------------------------------------------

data = pd.read_parquet(
    "../data/processed/razorshield_engineered_590k.parquet"
)

data = data.sort_values(
    "TransactionDT"
).reset_index(drop=True)

print("Data shape:", data.shape)


# ---------------------------------------------------------
# 2. Sort TRANSACTION data before merging
# ---------------------------------------------------------



# ---------------------------------------------------------
# 5. Free unnecessary objects
# ---------------------------------------------------------

# del db
# gc.collect()

# print("Memory cleanup completed.")

Data shape: (590540, 445)


In [9]:
lstm_features = [
    "log_TransactionAmt",
    "TransactionHour",
    "TransactionDay",
    "has_identity",
    "time_since_previous",
    "amount_change",
    "abs_amount_change",
    "card1_frequency",
    "card1_avg_previous_amount",
    "amount_vs_card_avg",
    "recent_card_transactions"
]

In [10]:
print("LSTM features:", len(lstm_features))


LSTM features: 11


In [11]:
X_lstm_raw = data[
    lstm_features
].astype(np.float32)

y_lstm_raw = data[
    "isFraud"
].astype(np.int8)

print("X:", X_lstm_raw.shape)
print("y:", y_lstm_raw.shape)

print(
    "Missing:",
    X_lstm_raw.isnull().sum().sum()
)

print(
    "Infinite:",
    np.isinf(X_lstm_raw).sum().sum()
)

X: (590540, 11)
y: (590540,)
Missing: 0
Infinite: 0


In [12]:
# ============================================================
# CHRONOLOGICAL 70 / 15 / 15 SPLIT
# ============================================================

split_1 = int(len(X_lstm_raw) * 0.70)
split_2 = int(len(X_lstm_raw) * 0.85)

X_train_lstm = X_lstm_raw.iloc[:split_1].copy()
X_val_lstm = X_lstm_raw.iloc[split_1:split_2].copy()
X_test_lstm = X_lstm_raw.iloc[split_2:].copy()

y_train_lstm = y_lstm_raw.iloc[:split_1].copy()
y_val_lstm = y_lstm_raw.iloc[split_1:split_2].copy()
y_test_lstm = y_lstm_raw.iloc[split_2:].copy()

print("Train:", X_train_lstm.shape)
print("Validation:", X_val_lstm.shape)
print("Test:", X_test_lstm.shape)

print("\nFraud counts:")
print("Train:", y_train_lstm.sum())
print("Validation:", y_val_lstm.sum())
print("Test:", y_test_lstm.sum())

Train: (413378, 11)
Validation: (88581, 11)
Test: (88581, 11)

Fraud counts:
Train: 14538
Validation: 3042
Test: 3083


In [13]:
from sklearn.preprocessing import StandardScaler

scaler_lstm = StandardScaler()

X_train_lstm_scaled = scaler_lstm.fit_transform(
    X_train_lstm
).astype(np.float32)

X_val_lstm_scaled = scaler_lstm.transform(
    X_val_lstm
).astype(np.float32)

X_test_lstm_scaled = scaler_lstm.transform(
    X_test_lstm
).astype(np.float32)

print("Train scaled:", X_train_lstm_scaled.shape)
print("Validation scaled:", X_val_lstm_scaled.shape)
print("Test scaled:", X_test_lstm_scaled.shape)

Train scaled: (413378, 11)
Validation scaled: (88581, 11)
Test scaled: (88581, 11)


In [14]:
SEQUENCE_LENGTH = 10

def create_sequences(X, y, sequence_length):

    X_seq = []
    y_seq = []

    for i in range(
        sequence_length,
        len(X)
    ):

        X_seq.append(
            X[
                i-sequence_length:i
            ]
        )

        y_seq.append(
            y[i]
        )

    return (
        np.asarray(X_seq, dtype=np.float32),
        np.asarray(y_seq, dtype=np.int8)
    )

In [15]:
X_train_seq, y_train_seq = create_sequences(
    X_train_lstm_scaled,
    y_train_lstm.to_numpy(),
    SEQUENCE_LENGTH
)

X_val_seq, y_val_seq = create_sequences(
    X_val_lstm_scaled,
    y_val_lstm.to_numpy(),
    SEQUENCE_LENGTH
)

X_test_seq, y_test_seq = create_sequences(
    X_test_lstm_scaled,
    y_test_lstm.to_numpy(),
    SEQUENCE_LENGTH
)

print("Train sequences:", X_train_seq.shape)
print("Validation sequences:", X_val_seq.shape)
print("Test sequences:", X_test_seq.shape)

Train sequences: (413368, 10, 11)
Validation sequences: (88571, 10, 11)
Test sequences: (88571, 10, 11)


In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

negative = np.sum(y_train_seq == 0)
positive = np.sum(y_train_seq == 1)

class_weight = {
    0: 1.0,
    1: negative / positive
}

print("Class weight:", class_weight)

Class weight: {0: 1.0, 1: np.float64(27.43362223139359)}


In [17]:
lstm_model = Sequential([
    
    LSTM(
        64,
        input_shape=(
            SEQUENCE_LENGTH,
            len(lstm_features)
        ),
        return_sequences=True
    ),

    Dropout(0.30),

    LSTM(
        32,
        return_sequences=False
    ),

    Dropout(0.30),

    Dense(
        16,
        activation="relu"
    ),

    Dropout(0.20),

    Dense(
        1,
        activation="sigmoid"
    )
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.AUC(
            name="roc_auc"
        ),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        ),
        tf.keras.metrics.Precision(
            name="precision"
        ),
        tf.keras.metrics.Recall(
            name="recall"
        )
    ]
)

lstm_model.summary()

e:\AI_Riskk\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 10, 64)         │        19,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,417 (126.63 KB)

 Trainable params: 32,417 (126.63 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
callbacks = [
    EarlyStopping(
        monitor="val_pr_auc",
        mode="max",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_pr_auc",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-6
    )
]

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,

    validation_data=(
        X_val_seq,
        y_val_seq
    ),

    epochs=30,
    batch_size=256,

    class_weight=class_weight,

    callbacks=callbacks,

    verbose=1
)

Epoch 1/30
1615/1615 ━━━━━━━━━━━━━━━━━━━━ 60s 31ms/step - loss: 1.3171 - pr_auc: 0.0513 - precision: 0.0437 - recall: 0.5140 - roc_auc: 0.5773 - val_loss: 0.7240 - val_pr_auc: 0.0574 - val_precision: 0.0420 - val_recall: 0.4896 - val_roc_auc: 0.5688 - learning_rate: 0.0010
Epoch 2/30
1615/1615 ━━━━━━━━━━━━━━━━━━━━ 47s 29ms/step - loss: 1.3104 - pr_auc: 0.0553 - precision: 0.0459 - recall: 0.4928 - roc_auc: 0.5875 - val_loss: 0.7584 - val_pr_auc: 0.0606 - val_precision: 0.0406 - val_recall: 0.6255 - val_roc_auc: 0.5821 - learning_rate: 0.0010
Epoch 3/30
1615/1615 ━━━━━━━━━━━━━━━━━━━━ 45s 28ms/step - loss: 1.3077 - pr_auc: 0.0562 - precision: 0.0468 - recall: 0.4838 - roc_auc: 0.5915 - val_loss: 0.7200 - val_pr_auc: 0.0616 - val_precision: 0.0448 - val_recall: 0.5002 - val_roc_auc: 0.5815 - learning_rate: 0.0010
Epoch 4/30
1615/1615 ━━━━━━━━━━━━━━━━━━━━ 48s 30ms/step - loss: 1.3050 - pr_auc: 0.0572 - precision: 0.0473 - recall: 0.4851 - roc_auc: 0.5943 - val_loss: 0.6631 - val_pr_auc: 0.

In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

lstm_test_prob = (
    lstm_model.predict(
        X_test_seq,
        batch_size=512,
        verbose=1
    ).ravel()
)

lstm_test_pred = (
    lstm_test_prob >= 0.5
).astype(int)

print("=" * 60)
print("RazorShield LSTM - FINAL TEST")
print("=" * 60)

print(
    f"Accuracy : {accuracy_score(y_test_seq, lstm_test_pred):.4f}"
)

print(
    f"Precision: {precision_score(y_test_seq, lstm_test_pred, zero_division=0):.4f}"
)

print(
    f"Recall   : {recall_score(y_test_seq, lstm_test_pred, zero_division=0):.4f}"
)

print(
    f"F1       : {f1_score(y_test_seq, lstm_test_pred, zero_division=0):.4f}"
)

print(
    f"ROC-AUC  : {roc_auc_score(y_test_seq, lstm_test_prob):.4f}"
)

print(
    f"PR-AUC   : {average_precision_score(y_test_seq, lstm_test_prob):.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test_seq,
        lstm_test_pred
    )
)

173/173 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step
RazorShield LSTM - FINAL TEST
Accuracy : 0.5323
Precision: 0.0406
Recall   : 0.5488
F1       : 0.0755
ROC-AUC  : 0.5619
PR-AUC   : 0.0454

Confusion Matrix:
[[45457 40031]
 [ 1391  1692]]


In [22]:
lstm_test_pred = (lstm_test_prob >= 0.5).astype(int)
# ============================================================
# LSTM VALIDATION THRESHOLD TUNING
# ============================================================

lstm_val_prob = (
    lstm_model.predict(
        X_val_seq,
        batch_size=512,
        verbose=1
    ).ravel()
)

threshold_results_lstm = []

for threshold in np.arange(0.05, 0.96, 0.05):

    pred = (
        lstm_val_prob >= threshold
    ).astype(int)

    threshold_results_lstm.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val_seq,
            pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_val_seq,
            pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_val_seq,
            pred,
            zero_division=0
        )
    })

threshold_df_lstm = pd.DataFrame(
    threshold_results_lstm
)

print(threshold_df_lstm)

173/173 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step
    threshold  precision    recall        f1
0        0.05   0.034334  1.000000  0.066389
1        0.10   0.034334  1.000000  0.066389
2        0.15   0.034334  1.000000  0.066389
3        0.20   0.034334  1.000000  0.066389
4        0.25   0.034334  1.000000  0.066389
5        0.30   0.034334  1.000000  0.066389
6        0.35   0.034329  0.999671  0.066378
7        0.40   0.034654  0.986846  0.066956
8        0.45   0.036838  0.828346  0.070539
9        0.50   0.044926  0.478132  0.082135
10       0.55   0.058140  0.282802  0.096450
11       0.60   0.076075  0.183821  0.107614
12       0.65   0.091303  0.124630  0.105395
13       0.70   0.115139  0.095035  0.104125
14       0.75   0.140909  0.071358  0.094739
15       0.80   0.165548  0.048668  0.075222
16       0.85   0.184154  0.028280  0.049031
17       0.90   0.250000  0.015455  0.029111
18       0.95   0.400000  0.003946  0.007815


In [23]:
# ============================================================
# SELECT BEST LSTM THRESHOLD
# ============================================================

best_row_lstm = threshold_df_lstm.loc[
    threshold_df_lstm["f1"].idxmax()
]

best_threshold_lstm = (
    best_row_lstm["threshold"]
)

print("=" * 60)
print("BEST LSTM THRESHOLD")
print("=" * 60)

print(
    "Threshold:",
    best_threshold_lstm
)

print(
    "Validation Precision:",
    round(best_row_lstm["precision"], 4)
)

print(
    "Validation Recall:",
    round(best_row_lstm["recall"], 4)
)

print(
    "Validation F1:",
    round(best_row_lstm["f1"], 4)
)

BEST LSTM THRESHOLD
Threshold: 0.6000000000000001
Validation Precision: 0.0761
Validation Recall: 0.1838
Validation F1: 0.1076


In [24]:
# ============================================================
# LSTM FINAL TEST - TUNED THRESHOLD
# ============================================================

lstm_test_pred_tuned = (
    lstm_test_prob >= best_threshold_lstm
).astype(int)

print("=" * 60)
print("RazorShield LSTM - FINAL TEST - TUNED")
print("=" * 60)

print(
    f"Accuracy : "
    f"{accuracy_score(y_test_seq, lstm_test_pred_tuned):.4f}"
)

print(
    f"Precision: "
    f"{precision_score(y_test_seq, lstm_test_pred_tuned, zero_division=0):.4f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_test_seq, lstm_test_pred_tuned, zero_division=0):.4f}"
)

print(
    f"F1       : "
    f"{f1_score(y_test_seq, lstm_test_pred_tuned, zero_division=0):.4f}"
)

print(
    f"ROC-AUC  : "
    f"{roc_auc_score(y_test_seq, lstm_test_prob):.4f}"
)

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_test_seq, lstm_test_prob):.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test_seq,
        lstm_test_pred_tuned
    )
)

RazorShield LSTM - FINAL TEST - TUNED
Accuracy : 0.8473
Precision: 0.0518
Recall   : 0.1956
F1       : 0.0819
ROC-AUC  : 0.5619
PR-AUC   : 0.0454

Confusion Matrix:
[[74444 11044]
 [ 2480   603]]


In [21]:
import os
os.makedirs(
    "../models",
    exist_ok=True
)

lstm_model.save(
    "../models/razorshield_lstm_590k.keras"
)

import joblib

joblib.dump(
    scaler_lstm,
    "../models/razorshield_lstm_scaler_590k.pkl"
)

joblib.dump(
    lstm_features,
    "../models/razorshield_lstm_features_590k.pkl"
)

joblib.dump(
    SEQUENCE_LENGTH,
    "../models/razorshield_lstm_sequence_length.pkl"
)

print("LSTM model and preprocessing artifacts saved.")

LSTM model and preprocessing artifacts saved.
